In [1]:
# ==============================================================================
# PHYSICS-INFORMED HYBRID FRAMEWORK FOR LEAK LOCALISATION (THE ULTIMATE MONOLITHIC SCRIPT)
# Verified Data: Van der Walt, Heyns, & Wilke (University of Pretoria, 2021)
# ==============================================================================

# ==============================================================================
# STEP 0: ENVIRONMENT SETUP & IMPORTS
# Installing required packages (PyWavelets for signal processing) and importing libraries
# ==============================================================================
import importlib, subprocess, sys
if importlib.util.find_spec('pywt') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'PyWavelets'])

import numpy as np
import pandas as pd
from scipy import signal, stats
import pywt
import re
from sklearn.linear_model import Ridge
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
import matplotlib.pyplot as plt
import matplotlib as mpl
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

# Setting up journal-friendly plot defaults (fonts, borders, resolution)
mpl.rcParams.update({
    'font.size': 10, 'font.family': 'DejaVu Sans',
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.dpi': 110, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
})

# ==============================================================================
# STEP 1: INTERACTIVE DATA INGESTION
# Prompting the user to upload the 9 .npy files directly into the Colab environment
# ==============================================================================
print("--- STEP 1: SYSTEM INITIALIZATION AND FILE UPLOAD ---")
print("PLEASE SELECT YOUR 9 .NPY FILES:")
uploaded = files.upload()

# Filtering only .npy files that contain 'eak' (Leak/leak) in their names to prevent wrong uploads
files_found = sorted([f for f in uploaded.keys() if f.endswith('.npy') and 'eak' in f.lower()])

assert len(files_found) == 9, f'CRITICAL ERROR: Expected 9 .npy files, but {len(files_found)} were uploaded.'
print(f"SUCCESS: 9 files successfully loaded. Commencing analysis...\n")

# ==============================================================================
# STEP 2: HYDRAULIC CONSTANTS & BASELINE DEFINITION
# Defining the experimental physical parameters and the Hazen-Williams analytic inversion
# ==============================================================================
FS = 10_000           # Sampling frequency in Hz
DURATION = 15         # Duration of measurement in seconds
C_HW = 100            # Hazen-Williams roughness coefficient
D_PIPE = 25e-3        # Pipe internal diameter in meters
L_PIPE = 65           # Total pipe length in meters
A_HW = 10.67 / (C_HW ** 1.852 * D_PIPE ** 4.8704)  # Hazen-Williams friction parameter

def analytic_leak_location(sample):
    """Calculates the theoretical leak location based on steady-state Hazen-Williams equation."""
    P1, Q1, P2, Q2 = sample[0], sample[1] / 1000.0, sample[2], sample[3] / 1000.0
    num = (P1 - P2) - A_HW * L_PIPE * Q2 ** 1.852
    den = A_HW * (Q1 ** 1.852 - Q2 ** 1.852)
    return num / den if abs(den) > 1e-12 else np.nan

# ==============================================================================
# STEP 3: BOOTSTRAP ANALYSIS (THE "PRECISELY WRONG" DIAGNOSIS)
# Evaluating the baseline model over randomly resampled windows to show its systematic bias
# ==============================================================================
print("--- STEP 2: PERFORMING BOOTSTRAP ANALYSIS ON ANALYTIC BASELINE ---")
rng = np.random.default_rng(0)
N_BOOT = 500          # Number of bootstrap iterations
BOOT_SIZE = 5000      # Samples per bootstrap iteration

bootstrap_results = []
summary_rows = []

for f in files_found:
    d = np.load(f)

    # Extracting the true location and leak size index using regular expressions
    nums = re.findall(r'\d+', f)
    loc_true = float(nums[-1]) if nums else 0.0
    size_idx = int(nums[0]) if len(nums) > 1 else 1
    case_name = f.replace('.npy', '')

    P1, Q1, P2, Q2 = d[:, 0], d[:, 1], d[:, 2], d[:, 3]
    summary_rows.append({
        'case': case_name, 'location_m': loc_true, 'size_idx': size_idx,
        'LL_analytic_raw': analytic_leak_location(d.mean(axis=0)),
        'QL': Q1.mean() - Q2.mean()
    })

    # Generating bootstrap samples and estimating confidence intervals
    preds = []
    for _ in range(N_BOOT):
        idx = rng.integers(0, d.shape[0], size=BOOT_SIZE)
        preds.append(analytic_leak_location(d[idx].mean(axis=0)))
    preds = np.array(preds)
    preds = preds[np.isfinite(preds)]
    ci_lo, ci_hi = np.percentile(preds, [2.5, 97.5])

    bootstrap_results.append({
        'case': case_name, 'location_m': loc_true,
        'boot_mean': preds.mean(), 'ci_lo': ci_lo, 'ci_hi': ci_hi,
    })

summary = pd.DataFrame(summary_rows)
summary['analytic_error'] = summary['LL_analytic_raw'] - summary['location_m']
bootstrap_df = pd.DataFrame(bootstrap_results)

# ==============================================================================
# STEP 4: HYBRID FEATURE EXTRACTION (33 FEATURES)
# Extracting physical, statistical, spectral, and wavelet features across 0.5s windows
# ==============================================================================
print("--- STEP 3: EXTRACTING 33-DIMENSIONAL MULTI-SCALE FEATURES ---")

def band_energy(x, fs, lo, hi, nperseg=None):
    """Calculates spectral energy within a specific frequency band using Welch's method."""
    x = x - x.mean()
    if nperseg is None: nperseg = min(2048, len(x))
    f, P = signal.welch(x, fs, nperseg=nperseg)
    m = (f >= lo) & (f <= hi)
    return float(np.trapezoid(P[m], f[m]))

def window_features(win):
    """Computes all 33 features for a given 0.5-second time window."""
    P1, Q1, P2, Q2 = win[:, 0], win[:, 1], win[:, 2], win[:, 3]
    dP = P1 - P2
    feats = {
        # Standard means and hydraulic prior
        'P1_mean': P1.mean(), 'Q1_mean': Q1.mean(), 'P2_mean': P2.mean(), 'Q2_mean': Q2.mean(),
        'dP_mean': dP.mean(), 'QL_mean': Q1.mean() - Q2.mean(),
        'LL_analytic': analytic_leak_location(win.mean(axis=0)),

        # Standard deviations (Noise / Turbulence)
        'P1_std': P1.std(), 'Q1_std': Q1.std(), 'P2_std': P2.std(), 'Q2_std': Q2.std(),

        # Higher-order statistics (Skewness and Kurtosis)
        'P1_skew': float(stats.skew(P1)), 'P1_kurt': float(stats.kurtosis(P1)),
        'P2_skew': float(stats.skew(P2)), 'P2_kurt': float(stats.kurtosis(P2)),
        'Q1_skew': float(stats.skew(Q1)), 'Q2_skew': float(stats.skew(Q2)),

        # Spectral energy features (Pump harmonics and high frequency)
        'E_P1_pump': band_energy(P1, FS, 45, 55), 'E_P1_100_500': band_energy(P1, FS, 100, 500),
        'E_P1_500_2k':  band_energy(P1, FS, 500, 2000), 'E_dP_100_500': band_energy(dP, FS, 100, 500),
        'E_dP_500_2k':  band_energy(dP, FS, 500, 2000),
    }

    # Wavelet decomposition (Daubechies-4) on the pressure differential
    dx = (dP - dP.mean())[::4]
    coeffs = pywt.wavedec(dx, 'db4', level=4)
    for i, c in enumerate(coeffs): feats[f'W_dP_lvl{i}'] = float(np.sum(c ** 2) / len(c))

    # Coherence between inlet and outlet pressure
    fc, Cxy = signal.coherence(P1 - P1.mean(), P2 - P2.mean(), FS, nperseg=min(2048, len(P1)))
    for lo, hi, nm in [(100, 500, 'mid'), (500, 2000, 'high')]:
        mask = (fc >= lo) & (fc <= hi)
        feats[f'coh_{nm}'] = float(Cxy[mask].mean()) if mask.any() else 0.0

    # Residuals mapping the error of the steady-state assumption
    Q1_m3 = Q1 / 1000.0
    res = dP - (A_HW * L_PIPE * Q1_m3 ** 1.852)
    feats['res_mean'] = res.mean(); feats['res_std'] = res.std(); feats['res_skew'] = float(stats.skew(res))
    return feats

# Slicing the full 15s record into 30 non-overlapping 0.5s windows
N_WINDOWS = 30
WIN_LEN = 150_000 // N_WINDOWS
all_rows = []

for f in files_found:
    d = np.load(f)
    nums = re.findall(r'\d+', f)
    loc_true = float(nums[-1]) if nums else 0.0
    case_name = f.replace('.npy', '')

    for w in range(N_WINDOWS):
        win = d[w * WIN_LEN : (w + 1) * WIN_LEN]
        feats = window_features(win)
        feats['case'] = case_name; feats['loc_true'] = loc_true; feats['ql_true'] = feats['QL_mean']
        all_rows.append(feats)

df = pd.DataFrame(all_rows)

# ==============================================================================
# STEP 5: LOOCV CROSS-VALIDATION & PLS-k3 MODEL TRAINING
# Training the Partial Least Squares model using Leave-One-Case-Out methodology
# ==============================================================================
print("--- STEP 4: TRAINING PLS-K3 MODEL VIA LEAVE-ONE-GROUP-OUT CV ---")
feature_cols = [c for c in df.columns if c not in ('case', 'loc_true', 'size_idx', 'ql_true')]
X = np.nan_to_num(df[feature_cols].values, nan=0, posinf=0, neginf=0)
y_loc = df['loc_true'].values
groups = df['case'].values

def run_loocv_model(estimator, X, y, groups):
    """Executes LOOCV iteratively and returns the window-level predictions."""
    preds_window = np.zeros(len(y))
    logo = LeaveOneGroupOut()
    for tr, te in logo.split(X, y, groups):
        scaler = StandardScaler().fit(X[tr])
        mdl = estimator().fit(scaler.transform(X[tr]), y[tr])
        p = mdl.predict(scaler.transform(X[te]))
        preds_window[te] = np.asarray(p).flatten()
    return preds_window

def aggregate_to_case(preds_window, y, groups):
    """Aggregates 30 window predictions into a single prediction per physical case."""
    cases = np.unique(groups)
    y_true = np.array([y[groups == c][0] for c in cases])
    y_pred = np.array([preds_window[groups == c].mean() for c in cases])
    return cases, y_true, y_pred

cases, y_true_case, _ = aggregate_to_case(np.zeros(len(y_loc)), y_loc, groups)
analytic_raw = np.array([df.loc[df['case'] == c, 'LL_analytic'].mean() for c in cases])

# Executing the PLS-k3 model
preds_pls = run_loocv_model(lambda: PLSRegression(n_components=3), X, y_loc, groups)
_, _, y_pred_pls = aggregate_to_case(preds_pls, y_loc, groups)
case_pred_std = np.array([preds_pls[groups == c].std() for c in cases])

def mae(pred, truth):  return np.mean(np.abs(pred - truth))
mae_analytic = mae(analytic_raw, y_true_case)
mae_hybrid = mae(y_pred_pls, y_true_case)
print(f"   => Analytic Baseline MAE : {mae_analytic:.2f} m")
print(f"   => Hybrid PLS-k3 MAE     : {mae_hybrid:.2f} m (Improvement Factor: {mae_analytic/mae_hybrid:.1f}x)\n")

# ==============================================================================
# STEP 6: CONFORMAL PREDICTION (UNCERTAINTY QUANTIFICATION)
# Generating calibrated confidence intervals using the Jackknife+ method
# ==============================================================================
print("--- STEP 5: CALCULATING JACKKNIFE+ CONFORMAL INTERVALS ---")
def jackknife_plus_coverage(alpha, y_pred, y_true):
    """Computes distribution-free conformal prediction bounds."""
    m = len(y_pred) - 1
    k = min(int(np.ceil((1 - alpha) * (m + 1))), m)
    abs_res = np.abs(y_pred - y_true)
    lo, hi, q_vals = [], [], []
    for i in range(len(y_pred)):
        others = np.delete(abs_res, i)
        q = np.sort(others)[k - 1]
        q_vals.append(q)
        lo.append(y_pred[i] - q)
        hi.append(y_pred[i] + q)
    lo, hi, q_vals = np.array(lo), np.array(hi), np.array(q_vals)
    covered = (y_true >= lo) & (y_true <= hi)
    return covered.mean(), lo, hi, q_vals

_, lo90, hi90, q90 = jackknife_plus_coverage(0.1, y_pred_pls, y_true_case)
_, lo80, hi80, q80 = jackknife_plus_coverage(0.2, y_pred_pls, y_true_case)

# ==============================================================================
# STEP 7: FIGURE GENERATION & AUTO-DOWNLOAD TO LOCAL PC
# Plotting the 7 manuscript figures, saving them as PNGs, and downloading them
# ==============================================================================
print("--- STEP 6: GENERATING, SAVING, AND DOWNLOADING ALL 7 FIGURES ---")
generated_files = [] # List to track saved images

# --- FIGURE 1: PRECISELY WRONG (Bootstrap Map) ---
fig1, ax1 = plt.subplots(figsize=(8, 4.5))
y_pos = np.arange(len(bootstrap_df))
for k, r in bootstrap_df.iterrows():
    ax1.errorbar(r['boot_mean'], k, xerr=[[r['boot_mean'] - r['ci_lo']], [r['ci_hi'] - r['boot_mean']]], fmt='o', color='#d62728', capsize=3, ms=6, elinewidth=1.2)
    ax1.scatter(r['location_m'], k, marker='s', s=80, facecolor='#2ca02c', edgecolor='black', zorder=3, linewidth=0.8)
ax1.set_yticks(y_pos); ax1.set_yticklabels([c.replace('Leak ', 'Leak_') for c in bootstrap_df['case']])
ax1.set_xlabel('Leak location estimate [m]'); ax1.set_title('Fig 1: Bootstrap 95% CI on analytic estimator (red) vs true location (green)\n"Precisely wrong": tight CIs, catastrophic bias')
ax1.axvline(0, color='grey', lw=0.3); ax1.axvline(L_PIPE, color='grey', lw=0.3, ls=':')
ax1.grid(axis='x', alpha=0.3); ax1.set_xlim(-65, 75)
plt.tight_layout()
fig1.savefig('fig1_precisely_wrong.png')
generated_files.append('fig1_precisely_wrong.png')
plt.close(fig1) # Closing plot to keep notebook clean

# --- FIGURE 2: BASELINE VS PROPOSED (Parity Plot) ---
fig2, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot([0, L_PIPE], [0, L_PIPE], 'k--', lw=0.8, alpha=0.6, label='Ideal (y = x)')
axes[0].scatter(y_true_case, analytic_raw, s=80, alpha=0.85, c='#d62728', marker='o', edgecolor='k', linewidth=0.5, label='Analytic H-W (raw)')
for t, p, c in zip(y_true_case, analytic_raw, cases):
    if abs(p - t) > 20: axes[0].annotate(c.replace('Leak ', 'Leak_'), (t, p), xytext=(6, -6), textcoords='offset points', fontsize=8)
axes[0].set_xlabel('True leak location [m]'); axes[0].set_ylabel('Predicted leak location [m]')
axes[0].set_title(f'Fig 2a: Analytic Hazen-Williams inversion\nMAE = {mae_analytic:.1f} m')
axes[0].legend(loc='upper left', frameon=False, fontsize=9); axes[0].set_xlim(0, L_PIPE); axes[0].set_ylim(-65, 70)
axes[0].axhline(0, color='grey', lw=0.3); axes[0].axhline(L_PIPE, color='grey', lw=0.3, ls=':')

axes[1].plot([0, L_PIPE], [0, L_PIPE], 'k--', lw=0.8, alpha=0.6, label='Ideal (y = x)')
axes[1].errorbar(y_true_case, y_pred_pls, yerr=2 * case_pred_std, fmt='o', markersize=8, color='#2ca02c', ecolor='#2ca02c', alpha=0.85, capsize=3, markeredgecolor='k', markeredgewidth=0.5, label='Hybrid PLS-k3 (±2σ within-case)')
axes[1].set_xlabel('True leak location [m]'); axes[1].set_ylabel('Predicted leak location [m]')
axes[1].set_title(f'Fig 2b: Physics-informed hybrid model\nMAE = {mae_hybrid:.2f} m')
axes[1].legend(loc='upper left', frameon=False, fontsize=9); axes[1].set_xlim(0, L_PIPE); axes[1].set_ylim(0, L_PIPE)
plt.tight_layout()
fig2.savefig('fig2_baseline_vs_proposed.png')
generated_files.append('fig2_baseline_vs_proposed.png')
plt.close(fig2)

# --- FIGURE 3: ERROR BUDGET DECOMPOSITION (Bias vs Variance) ---
bias_sq = (y_pred_pls - y_true_case) ** 2
variance = case_pred_std ** 2
ql_per_case = np.array([df.loc[df['case'] == c, 'ql_true'].mean() for c in cases])
order = np.lexsort((ql_per_case, y_true_case))

fig3, ax3 = plt.subplots(figsize=(9, 4.5))
x_pos = np.arange(len(cases))
ax3.bar(x_pos, bias_sq[order], width=0.7, color='#d62728', label='Bias² (model mis-specification)', alpha=0.85, edgecolor='k', linewidth=0.5)
ax3.bar(x_pos, variance[order], bottom=bias_sq[order], width=0.7, color='#1f77b4', label='Variance (window spread)', alpha=0.85, edgecolor='k', linewidth=0.5)
labels = [f"{cases[i].replace('Leak ', 'Leak_')}\n{ql_per_case[i]:.2f} l/s" for i in order]
ax3.set_xticks(x_pos); ax3.set_xticklabels(labels, fontsize=8)
ax3.set_ylabel('Error budget [m²]'); ax3.set_title('Fig 3: Per-case error decomposition — Hybrid PLS-k3 model')
ax3.legend(loc='upper right', frameon=False); ax3.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig3.savefig('fig3_error_budget.png')
generated_files.append('fig3_error_budget.png')
plt.close(fig3)

# --- FIGURE 4: CONFORMAL UQ (Jackknife+ intervals) ---
fig4, ax4 = plt.subplots(figsize=(8, 5))
ax4.plot([0, L_PIPE], [0, L_PIPE], 'k--', lw=0.8, alpha=0.6)
for i in range(len(cases)):
    t = y_true_case[i]; p = y_pred_pls[i]
    ax4.errorbar(t, p, yerr=[[p - lo90[i]], [hi90[i] - p]], fmt='none', ecolor='#2ca02c', elinewidth=1.0, capsize=0, alpha=0.35)
    ax4.errorbar(t, p, yerr=[[p - lo80[i]], [hi80[i] - p]], fmt='none', ecolor='#2ca02c', elinewidth=2.0, capsize=3, alpha=0.75)
    ax4.scatter(t, p, s=70, c='#2ca02c', marker='o', edgecolor='black', linewidth=0.6, zorder=4)

from matplotlib.lines import Line2D
custom = [
    Line2D([0],[0], color='k', ls='--', lw=0.8, label='Ideal (y = x)'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#2ca02c', markersize=8, markeredgecolor='k', markeredgewidth=0.5, label='Point prediction'),
    Line2D([0],[0], color='#2ca02c', lw=2.0, alpha=0.75, label='80% conformal interval'),
    Line2D([0],[0], color='#2ca02c', lw=1.0, alpha=0.35, label='90% conformal interval'),
]
ax4.legend(handles=custom, loc='upper left', frameon=False, fontsize=9)
n_cov_90 = int(jackknife_plus_coverage(0.1, y_pred_pls, y_true_case)[0] * len(cases))
n_cov_80 = int(jackknife_plus_coverage(0.2, y_pred_pls, y_true_case)[0] * len(cases))
ax4.set_xlabel('True leak location [m]'); ax4.set_ylabel('Predicted leak location [m]')
ax4.set_title(f'Fig 4: Hybrid PLS-k3 with jackknife+ conformal intervals\n90% nominal: {n_cov_90}/{len(cases)} covered; 80% nominal: {n_cov_80}/{len(cases)} covered')
ax4.set_xlim(0, L_PIPE); ax4.set_ylim(-10, 75); ax4.grid(alpha=0.3)
plt.tight_layout()
fig4.savefig('fig4_conformal_uq.png')
generated_files.append('fig4_conformal_uq.png')
plt.close(fig4)

# --- FIGURE 5: FEATURE IMPORTANCE (Model interpretability) ---
scaler_full = StandardScaler().fit(X)
pls_full = PLSRegression(n_components=3).fit(scaler_full.transform(X), y_loc)
coefs_abs = np.abs(pls_full.coef_).flatten()

top_idx = np.argsort(coefs_abs)[-15:]
top_feats = [feature_cols[i] for i in top_idx]
top_vals = coefs_abs[top_idx]

color_map = []
for f in top_feats:
    if 'analytic' in f or 'res' in f or 'QL' in f: color_map.append('#ff7f0e')
    elif 'W_dP' in f: color_map.append('#9467bd')
    elif 'E_' in f: color_map.append('#2ca02c')
    else: color_map.append('#17becf')

fig5, ax5 = plt.subplots(figsize=(10, 6))
bars = ax5.barh(np.arange(len(top_feats)), top_vals, color=color_map, edgecolor='k', linewidth=0.5)
ax5.set_yticks(np.arange(len(top_feats))); ax5.set_yticklabels(top_feats, fontsize=10)
ax5.set_xlabel('Mean |PLS coefficient| across LOOCV folds', fontsize=12)
ax5.set_title('Fig 5: Feature importance — hybrid physics + signal features', fontsize=12)

custom_lines_5 = [Line2D([0], [0], color='#ff7f0e', lw=6), Line2D([0], [0], color='#9467bd', lw=6),
                  Line2D([0], [0], color='#2ca02c', lw=6), Line2D([0], [0], color='#17becf', lw=6)]
ax5.legend(custom_lines_5, ['Physics-model derived', 'Wavelet multi-scale', 'Spectral band energy', 'Higher-order statistics'], loc='lower right', frameon=False)
plt.tight_layout()
fig5.savefig('fig5_feature_importance.png')
generated_files.append('fig5_feature_importance.png')
plt.close(fig5)

# --- FIGURE 6: TIME SERIES & PSD (Signal analysis for the first dataset) ---
sample_file = files_found[0]
d_sample = np.load(sample_file)
t_3s = np.arange(30000) / FS
f_welch, psd_p1 = signal.welch(d_sample[:, 0] - d_sample[:, 0].mean(), FS, nperseg=4096)
_, psd_q1 = signal.welch(d_sample[:, 1] - d_sample[:, 1].mean(), FS, nperseg=4096)
_, psd_p2 = signal.welch(d_sample[:, 2] - d_sample[:, 2].mean(), FS, nperseg=4096)
_, psd_q2 = signal.welch(d_sample[:, 3] - d_sample[:, 3].mean(), FS, nperseg=4096)

fig6 = plt.figure(figsize=(14, 8))
fig6.suptitle(f'Fig 6: Case {sample_file.replace(".npy","")}: time series (3 s) and Welch PSDs (15 s)', fontsize=12)

for i, (ts, psd, ylabel, c) in enumerate(zip(
    [d_sample[:30000, 0], d_sample[:30000, 1], d_sample[:30000, 2], d_sample[:30000, 3]],
    [psd_p1, psd_q1, psd_p2, psd_q2],
    ['P1 [m]', 'Q1 [l/s]', 'P2 [m]', 'Q2 [l/s]'],
    ['#1f77b4', '#ff7f0e', '#1f77b4', '#ff7f0e']
)):
    ax_t = plt.subplot(4, 4, (i*4+1, i*4+3))
    ax_t.plot(t_3s, ts, color=c, lw=0.5); ax_t.set_ylabel(ylabel); ax_t.set_xlim(0, 3); ax_t.grid(alpha=0.3)
    if i == 3: ax_t.set_xlabel('Time [s]')
    else: ax_t.set_xticklabels([])

    ax_p = plt.subplot(4, 4, i*4+4)
    ax_p.loglog(f_welch, psd, color=c, lw=0.8); ax_p.set_ylabel('PSD'); ax_p.set_xlim(1, 5000); ax_p.grid(alpha=0.3, which='both')
    if i == 0:
        ax_p.axvline(48.8, color='red', ls=':', lw=0.8, alpha=0.6); ax_p.text(55, 1e-6, 'pump 48.8 Hz', color='red', fontsize=6, rotation=90)
    if i == 3: ax_p.set_xlabel('Frequency [Hz]')
    else: ax_p.set_xticklabels([])

plt.tight_layout(); plt.subplots_adjust(top=0.92)
fig6.savefig('fig6_timeseries_psd.png')
generated_files.append('fig6_timeseries_psd.png')
plt.close(fig6)

# --- FIGURE 7: PERFORMANCE MAP (Error matrix heatmap) ---
err_analytic = np.abs(analytic_raw - y_true_case)
err_hybrid = np.abs(y_pred_pls - y_true_case)
locs = [15.0, 30.0, 50.0]

grid_analytic = np.zeros((3, 3))
grid_hybrid = np.zeros((3, 3))

for i in range(len(cases)):
    l_idx = locs.index(y_true_case[i])
    s_val = df.loc[df['case'] == cases[i], 'ql_true'].mean()
    s_idx = 0 if s_val < 0.2 else (1 if s_val < 0.5 else 2)
    grid_analytic[s_idx, l_idx] = err_analytic[i]
    grid_hybrid[s_idx, l_idx] = err_hybrid[i]

fig7, axes7 = plt.subplots(1, 2, figsize=(12, 4.5))

im1 = axes7[0].imshow(grid_analytic, cmap='Reds', aspect='auto', vmin=0, vmax=10)
axes7[0].set_title('Fig 7a: Analytic baseline (clipped)\nabsolute error [m]')
axes7[0].set_xticks([0, 1, 2]); axes7[0].set_xticklabels([15, 30, 50])
axes7[0].set_yticks([0, 1, 2]); axes7[0].set_yticklabels(['Small\n~0.1', 'Medium\n~0.3', 'Large\n~0.6'])
axes7[0].set_xlabel('True leak location [m]'); axes7[0].set_ylabel('Leak size [l/s]')
for i in range(3):
    for j in range(3):
        color = "white" if grid_analytic[i, j] > 5 else "black"
        axes7[0].text(j, i, f"{grid_analytic[i, j]:.1f}", ha="center", va="center", color=color)
plt.colorbar(im1, ax=axes7[0], label='|error| [m]', fraction=0.046, pad=0.04)

im2 = axes7[1].imshow(grid_hybrid, cmap='Greens', aspect='auto', vmin=0, vmax=10)
axes7[1].set_title('Fig 7b: Proposed (|bias|)\nabsolute error [m]')
axes7[1].set_xticks([0, 1, 2]); axes7[1].set_xticklabels([15, 30, 50])
axes7[1].set_yticks([0, 1, 2]); axes7[1].set_yticklabels(['Small\n~0.1', 'Medium\n~0.3', 'Large\n~0.6'])
axes7[1].set_xlabel('True leak location [m]'); axes7[1].set_ylabel('Leak size [l/s]')
for i in range(3):
    for j in range(3):
        color = "white" if grid_hybrid[i, j] > 5 else "black"
        axes7[1].text(j, i, f"{grid_hybrid[i, j]:.1f}", ha="center", va="center", color=color)
plt.colorbar(im2, ax=axes7[1], label='|error| [m]', fraction=0.046, pad=0.04)

plt.tight_layout()
fig7.savefig('fig7_performance_map.png')
generated_files.append('fig7_performance_map.png')
plt.close(fig7)

# ==============================================================================
# STEP 8: AUTO-DOWNLOAD ALL FIGURES
# Triggers browser downloads for all generated image files
# ==============================================================================
print(f"--- DOWNLOADING {len(generated_files)} FIGURES TO YOUR LOCAL COMPUTER ---")
for img_file in generated_files:
    print(f"Downloading: {img_file}")
    files.download(img_file)

print("\n--- ANALYSIS AND DOWNLOADING COMPLETED SUCCESSFULLY ---")

--- STEP 1: SYSTEM INITIALIZATION AND FILE UPLOAD ---
PLEASE SELECT YOUR 9 .NPY FILES:


Saving Leak 1_15m.npy to Leak 1_15m.npy
Saving Leak 1_30m.npy to Leak 1_30m.npy
Saving Leak 1_50m.npy to Leak 1_50m.npy
Saving Leak 2_15m.npy to Leak 2_15m.npy
Saving Leak 2_30m.npy to Leak 2_30m.npy
Saving Leak 2_50m.npy to Leak 2_50m.npy
Saving Leak 3_15m.npy to Leak 3_15m.npy
Saving Leak 3_30m.npy to Leak 3_30m.npy
Saving Leak 3_50m.npy to Leak 3_50m.npy
SUCCESS: 9 files successfully loaded. Commencing analysis...

--- STEP 2: PERFORMING BOOTSTRAP ANALYSIS ON ANALYTIC BASELINE ---
--- STEP 3: EXTRACTING 33-DIMENSIONAL MULTI-SCALE FEATURES ---
--- STEP 4: TRAINING PLS-K3 MODEL VIA LEAVE-ONE-GROUP-OUT CV ---
   => Analytic Baseline MAE : 14.84 m
   => Hybrid PLS-k3 MAE     : 2.64 m (Improvement Factor: 5.6x)

--- STEP 5: CALCULATING JACKKNIFE+ CONFORMAL INTERVALS ---
--- STEP 6: GENERATING, SAVING, AND DOWNLOADING ALL 7 FIGURES ---
--- DOWNLOADING 7 FIGURES TO YOUR LOCAL COMPUTER ---
Downloading: fig1_precisely_wrong.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: fig2_baseline_vs_proposed.png


...

<IPython.core.display.Javascript object>

Downloading: fig7_performance_map.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


--- ANALYSIS AND DOWNLOADING COMPLETED SUCCESSFULLY ---
